In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "gpu")
print("Device:", device)

Device: cuda


In [3]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),
                         (0.2023,0.1994,0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),
                         (0.2023,0.1994,0.2010))
])

In [4]:
train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform_train
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform_test
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

classes = train_dataset.classes
print(classes)

100%|██████████| 170M/170M [00:13<00:00, 12.4MB/s]


['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [5]:
class CNN(nn.Module):

    def __init__(self):
        super(CNN,self).__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.Conv2d(256,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Linear(256*4*4,512),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(512,10)
        )

    def forward(self,x):

        x = self.features(x)

        x = x.view(x.size(0),-1)

        x = self.classifier(x)

        return x

In [6]:
model = CNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer,
                                            step_size=20,
                                            gamma=0.1)

In [7]:
num_epochs = 30

train_acc_list = []
train_loss_list = []

for epoch in range(num_epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    scheduler.step()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    train_loss_list.append(epoch_loss)
    train_acc_list.append(epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | Loss {epoch_loss:.4f} | Acc {epoch_acc:.2f}%")

Epoch 1/30 | Loss 1.7493 | Acc 34.41%
Epoch 2/30 | Loss 1.2772 | Acc 53.84%
Epoch 3/30 | Loss 1.0679 | Acc 62.27%
Epoch 4/30 | Loss 0.9345 | Acc 67.36%
Epoch 5/30 | Loss 0.8384 | Acc 71.21%
Epoch 6/30 | Loss 0.7512 | Acc 74.60%
Epoch 7/30 | Loss 0.6860 | Acc 76.92%
Epoch 8/30 | Loss 0.6285 | Acc 79.34%
Epoch 9/30 | Loss 0.5795 | Acc 81.03%
Epoch 10/30 | Loss 0.5367 | Acc 82.28%
Epoch 11/30 | Loss 0.4999 | Acc 83.59%
Epoch 12/30 | Loss 0.4730 | Acc 84.53%
Epoch 13/30 | Loss 0.4421 | Acc 85.58%
Epoch 14/30 | Loss 0.4168 | Acc 86.54%
Epoch 15/30 | Loss 0.3930 | Acc 87.20%
Epoch 16/30 | Loss 0.3728 | Acc 87.98%
Epoch 17/30 | Loss 0.3485 | Acc 88.58%
Epoch 18/30 | Loss 0.3368 | Acc 88.84%
Epoch 19/30 | Loss 0.3170 | Acc 89.71%
Epoch 20/30 | Loss 0.3040 | Acc 90.04%
Epoch 21/30 | Loss 0.2271 | Acc 92.59%
Epoch 22/30 | Loss 0.2018 | Acc 93.30%
Epoch 23/30 | Loss 0.1931 | Acc 93.56%
Epoch 24/30 | Loss 0.1863 | Acc 93.85%
Epoch 25/30 | Loss 0.1804 | Acc 94.01%
Epoch 26/30 | Loss 0.1721 | Acc 94

In [8]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total

print("Test Accuracy:", test_accuracy)

Test Accuracy: 90.47
